# Модуль 7. Файлы, исключения, JSON/CSV
## Курс «Программирование на Python с нуля»

**Для магистрантов 1 курса направления «Инструменты ИИ в изучении английского языка»**

Занятия 19–21 (12 академических часов: 4 ч теории + 8 ч практики)

**Среда:** Google Colab  
**Язык:** русский  
**Пререквизиты:** модули 1–6 (переменные, условия, циклы, строки, списки, словари, множества, функции, модули)

> **Структура блокнота:**
> - Занятие 19: работа с текстовыми файлами.
> - Занятие 20: исключения и формат JSON.
> - Занятие 21: формат CSV и практикум.
>
> Все практические задания снабжены автопроверками. Домашние задания и задания «со звёздочкой» — в конце.

## Цели модуля

После этого модуля вы сможете:

- сохранять данные в текстовые файлы и читать их обратно;
- работать с режимами `r`, `w`, `a`, конструкцией `with`, кодировкой UTF-8;
- обрабатывать ошибки через `try / except / else / finally`;
- создавать собственные исключения через `raise`;
- сериализовать и десериализовать данные в JSON (`json.dump`, `json.load`, `json.dumps`, `json.loads`);
- читать и записывать таблицы в формате CSV (`csv.reader`, `csv.writer`, `csv.DictReader`, `csv.DictWriter`);
- строить приложения с сохранением состояния между запусками;
- избегать типичных ошибок: забытый `close`, потерянные данные, неверные пути, некорректные кодировки.

## 🛠 Утилита для автопроверок

Следующая ячейка определяет вспомогательные функции. **Запустите её один раз перед началом практики.**

In [ ]:
# Утилиты для автоматической проверки заданий.
# Запустите эту ячейку один раз.

import os
import json as _json


def check_value(name, actual, expected):
    """Проверяет, что переменная равна ожидаемому значению."""
    if actual == expected:
        print(f"✅ Задание пройдено: {name} = {actual!r}")
        return True
    else:
        print(f"❌ Ошибка: ожидалось {name} = {expected!r}, получено {actual!r}")
        return False


def check_file_exists(path):
    """Проверяет, что файл существует."""
    if os.path.exists(path):
        print(f"✅ Файл существует: {path}")
        return True
    else:
        print(f"❌ Файл не найден: {path}")
        return False


def check_file_content(path, expected_lines):
    """Сравнивает содержимое файла со списком ожидаемых строк."""
    if not os.path.exists(path):
        print(f"❌ Файл не найден: {path}")
        return False
    with open(path, "r", encoding="utf-8") as f:
        actual = [line.rstrip("\n") for line in f.readlines()]
    if actual == expected_lines:
        print(f"✅ Содержимое файла {path} совпадает с ожидаемым")
        return True
    else:
        print(f"❌ Содержимое файла {path} не совпадает")
        print(f"   Ожидалось: {expected_lines}")
        print(f"   Получено:  {actual}")
        return False


def check_json_file(path, expected_obj):
    """Проверяет, что JSON-файл содержит ожидаемый объект."""
    if not os.path.exists(path):
        print(f"❌ Файл не найден: {path}")
        return False
    try:
        with open(path, "r", encoding="utf-8") as f:
            actual = _json.load(f)
    except Exception as e:
        print(f"❌ Ошибка чтения JSON: {type(e).__name__}: {e}")
        return False
    if actual == expected_obj:
        print(f"✅ JSON-файл {path} корректен")
        return True
    else:
        print(f"❌ JSON-файл {path} не совпадает с ожидаемым")
        print(f"   Ожидалось: {expected_obj}")
        print(f"   Получено:  {actual}")
        return False


print("Утилиты автопроверки загружены.")

Утилиты автопроверки загружены.


## 🗂 Подготовка учебных файлов

Эта ячейка создаёт файлы, с которыми мы будем работать. **Запустите её один раз.**

In [ ]:
# Создаём учебные файлы
with open("sample_text.txt", "w", encoding="utf-8") as f:
    f.write("AI helps learn English\n")
    f.write("Python is a programming language\n")
    f.write("Practice makes perfect\n")

with open("scores.csv", "w", encoding="utf-8") as f:
    f.write("name,score\n")
    f.write("Anna,85\n")
    f.write("Ivan,72\n")
    f.write("Olga,91\n")

print("Файлы sample_text.txt и scores.csv созданы.")

Файлы sample_text.txt и scores.csv созданы.


---

# Часть 1. Работа с файлами

## 1.1. Зачем сохранять данные в файлы

Переменные живут только пока работает программа. Как только вы закрываете Colab или перезапускаете ядро — все данные исчезают. Чтобы сохранить результаты между запусками, их записывают **в файлы**.

**Где это нужно:**

- дневник студента, который накапливает записи;
- словарь английских слов, который пополняется;
- результаты теста, которые нужно сохранить для преподавателя;
- настройки приложения.

**Файлы бывают:**

| Тип | Пример | Формат |
|-----|--------|--------|
| Текстовый | `notes.txt` | Любой текст |
| JSON | `data.json` | Структурированные данные |
| CSV | `table.csv` | Таблица |
| Бинарный | `image.png` | Байты (не для этого курса) |

Мы разберём первые три.

## 1.2. Функция `open()` и режимы

### 1.2.1. Синтаксис

    f = open("file.txt", "r", encoding="utf-8")
    content = f.read()
    f.close()

`open(путь, режим, encoding)` — открывает файл и возвращает объект-файл.

### 1.2.2. Режимы открытия

| Режим | Что делает | Если файла нет |
|-------|------------|----------------|
| `r` | Только чтение | `FileNotFoundError` |
| `w` | Запись (перезаписывает!) | Создаётся |
| `a` | Дописывание в конец | Создаётся |
| `x` | Запись только если файла нет | `FileExistsError` |
| `r+` | Чтение и запись | `FileNotFoundError` |

**Опасно:** режим `w` **полностью перезаписывает** файл. Если файл был — его содержимое исчезает без предупреждения.

### 1.2.3. Кодировка

Всегда указывайте `encoding="utf-8"` — это избавит от проблем с русскими буквами и спецсимволами на разных системах.

## 1.3. Конструкция `with` — безопасное открытие

Ручное `close()` — плохая практика: если в коде произойдёт ошибка, `close()` не выполнится, файл останется открытым. Правильный способ — `with`:

    with open("file.txt", "r", encoding="utf-8") as f:
        content = f.read()

**Что делает `with`:**

- открывает файл;
- выполняет тело блока;
- **автоматически закрывает файл**, даже если внутри была ошибка.

**Правило:** всегда используйте `with open(...) as f:` — это самый надёжный способ.

In [ ]:
# Пример 1. Чтение файла через with
with open("sample_text.txt", "r", encoding="utf-8") as f:
    content = f.read()

print("Содержимое файла:")
print(content)
print("Тип:", type(content).__name__)
print("Длина в символах:", len(content))

Содержимое файла:
AI helps learn EnglishPython is a programming languagePractice makes perfect
Тип: str
Длина в символах: 76


## 1.4. Чтение файла: четыре способа

### 1.4.1. `read()` — весь файл целиком в одну строку

    with open("file.txt", encoding="utf-8") as f:
        text = f.read()      # одна большая строка

Подходит для небольших файлов.

### 1.4.2. `readlines()` — список строк

    with open("file.txt", encoding="utf-8") as f:
        lines = f.readlines()   # ['строка1\n', 'строка2\n', ...]

Обратите внимание: в конце каждой строки остаётся символ `\n`. Убирается через `.strip()` или `.rstrip()`.

### 1.4.3. `readline()` — одна строка за вызов

    with open("file.txt", encoding="utf-8") as f:
        line1 = f.readline()   # первая
        line2 = f.readline()   # вторая

### 1.4.4. Итерация по файлу — самый эффективный способ

    with open("file.txt", encoding="utf-8") as f:
        for line in f:
            print(line.strip())

Этот способ не загружает весь файл в память сразу — работает даже для очень больших файлов.

In [ ]:
# Пример 2. Четыре способа чтения
# Способ 1: read()
with open("sample_text.txt", encoding="utf-8") as f:
    text = f.read()
print("read(): длина =", len(text))

# Способ 2: readlines()
with open("sample_text.txt", encoding="utf-8") as f:
    lines = f.readlines()
print("readlines(): количество =", len(lines))
print("Первая строка (repr):", repr(lines[0]))

# Способ 3: итерация
with open("sample_text.txt", encoding="utf-8") as f:
    for line in f:
        print("  →", line.strip())

read(): длина = 79
readlines(): количество = 3
Первая строка (repr): 'AI helps learn English\n'
  → AI helps learn English
  → Python is a programming language
  → Practice makes perfect


## 1.5. Запись в файл

### 1.5.1. Режим `w` — перезапись

    with open("file.txt", "w", encoding="utf-8") as f:
        f.write("Строка 1\n")
        f.write("Строка 2\n")

**Важно:** `write()` **не добавляет** перенос строки сам — ставьте `\n` вручную.

### 1.5.2. Режим `a` — дописывание

    with open("file.txt", "a", encoding="utf-8") as f:
        f.write("Новая строка\n")

Файл не очищается, новая запись добавляется в конец.

### 1.5.3. `writelines()` — запись списка строк

    lines = ["Строка 1\n", "Строка 2\n"]
    with open("file.txt", "w", encoding="utf-8") as f:
        f.writelines(lines)

Обратите внимание: `writelines` **не добавляет** переносы между строками.

In [ ]:
# Пример 3. Запись и дописывание
# Перезаписываем файл
with open("notes.txt", "w", encoding="utf-8") as f:
    f.write("Запись 1\n")
    f.write("Запись 2\n")

print("После записи:")
with open("notes.txt", encoding="utf-8") as f:
    print(f.read())

# Дописываем
with open("notes.txt", "a", encoding="utf-8") as f:
    f.write("Запись 3\n")

print("После дописывания:")
with open("notes.txt", encoding="utf-8") as f:
    print(f.read())

После записи:
Запись 1
Запись 2

После дописывания:
Запись 1
Запись 2
Запись 3



## Практика 1. Записать список в файл

**Описание:** дан список слов. Запишите каждое слово в файл `words_out.txt` на отдельной строке. Используйте режим `w`.

**Ожидаемый результат:**
```
✅ Содержимое файла words_out.txt совпадает с ожидаемым
```

In [ ]:
# Практика 1. Запись списка слов в файл
words = ["apple", "banana", "cherry"]

# Введите ваш код ниже:
# YOUR CODE HERE


# --- Автопроверка ---
check_file_content("words_out.txt", ["apple", "banana", "cherry"])

<details>
<summary>Подсказка</summary>

Откройте файл в режиме `w` через `with`, пройдитесь `for w in words:` и в теле напишите `f.write(w + "\n")`.

</details>

## Практика 2. Подсчитать строки в файле

**Описание:** откройте файл `sample_text.txt` и посчитайте в нём количество строк. Сохраните результат в переменную `line_count`.

**Ожидаемый результат:**
```
✅ Задание пройдено: line_count = 3
```

In [ ]:
# Практика 2. Подсчёт строк
line_count = 0

# Введите ваш код ниже:
# YOUR CODE HERE


# --- Автопроверка ---
check_value("line_count", line_count, 3)

## Практика 3. Дописать строку в файл

**Описание:** допишите в файл `sample_text.txt` новую строку `"New line"` (с переводом строки в конце). Используйте режим `a`.

**Ожидаемый результат:**
```
✅ Содержимое файла sample_text.txt совпадает с ожидаемым
```

In [ ]:
# Практика 3. Дописывание
# Введите ваш код ниже:
# YOUR CODE HERE


# --- Автопроверка ---
check_file_content(
    "sample_text.txt",
    [
        "AI helps learn English",
        "Python is a programming language",
        "Practice makes perfect",
        "New line",
    ],
)

## Практика 4. Подсчитать слова в файле

**Описание:** прочитайте файл `sample_text.txt` и посчитайте **общее количество слов** во всех строках. Сохраните результат в переменную `word_count`.

**Ожидаемый результат:** (после практики 3 файл содержит 4 строки, из них 16 слов)
```
✅ Задание пройдено: word_count = 16
```

In [ ]:
# Практика 4. Подсчёт слов
word_count = 0

# Введите ваш код ниже:
# YOUR CODE HERE


# --- Автопроверка ---
check_value("word_count", word_count, 16)

<details>
<summary>Подсказка</summary>

Пройдитесь по файлу в цикле `for line in f:`, для каждой строки делайте `word_count += len(line.split())`.

</details>

---

# Часть 2. Исключения

## 2.1. Что такое исключение

**Исключение** (exception) — это ошибка, которая возникает во время выполнения программы. Если её не обработать, программа останавливается.

### Основные типы исключений

| Исключение | Когда возникает |
|------------|------------------|
| `FileNotFoundError` | Файл не найден |
| `ValueError` | Неверное значение (`int("abc")`) |
| `TypeError` | Неверный тип (`"a" + 1`) |
| `KeyError` | Ключ не найден в словаре |
| `IndexError` | Индекс вне диапазона |
| `ZeroDivisionError` | Деление на ноль |
| `PermissionError` | Нет прав на файл |
| `JSONDecodeError` | Некорректный JSON |

**Правило:** исключение — это не «баг в Python», а **сигнал о проблеме**, которую нужно обработать.

## 2.2. `try / except` — обработка ошибок

### 2.2.1. Синтаксис

    try:
        # код, который может вызвать ошибку
        result = 10 / 0
    except ZeroDivisionError:
        # что делать при ошибке
        print("На ноль делить нельзя")

### 2.2.2. Как это работает

1. Python пытается выполнить код в `try`.
2. Если ошибки нет — блок `except` пропускается.
3. Если ошибка есть — Python **ищет подходящий `except`** и выполняет его.
4. Программа **продолжает работу** после блока `try/except`.

### 2.2.3. Несколько `except`

    try:
        x = int(input("Введите число: "))
        result = 10 / x
    except ValueError:
        print("Это не число")
    except ZeroDivisionError:
        print("Деление на ноль")

**Важно:** порядок `except` важен — более специфичные должны идти раньше.

### 2.2.4. Ловим всё подряд — плохая идея

    try:
        ...
    except Exception:
        pass       # ❌ так делать не надо

Такой код **скрывает все ошибки**, включая опечатки в именах переменных. Используйте только тогда, когда действительно нужно обработать любую ошибку (например, в долгоживущем приложении).

In [ ]:
# Пример 4. try/except
def safe_divide(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        print("Ошибка: деление на ноль")
        return None

print(safe_divide(10, 2))
print(safe_divide(10, 0))

5.0
Ошибка: деление на ноль
None


In [ ]:
# Пример 5. Несколько except
def parse_number(s):
    try:
        return int(s)
    except ValueError:
        print(f"'{s}' — это не число")
        return None

print(parse_number("42"))
print(parse_number("hello"))
print(parse_number("3.14"))   # тоже ValueError

42
'hello' — это не число
None
'3.14' — это не число
None


## 2.3. `else` и `finally`

### 2.3.1. `else` — если ошибки НЕ было

    try:
        x = int("42")
    except ValueError:
        print("Ошибка")
    else:
        print("Всё хорошо, x =", x)

### 2.3.2. `finally` — выполняется всегда

    try:
        f = open("file.txt")
    except FileNotFoundError:
        print("Файл не найден")
    finally:
        print("Этот блок выполнится в любом случае")

`finally` используют для **освобождения ресурсов**: закрытие файлов, соединений с БД. В нашем курсе чаще всего это не нужно — `with` делает это автоматически.

In [ ]:
# Пример 6. try/except/else/finally
def full_demo(s):
    try:
        x = int(s)
    except ValueError:
        print(f"Ошибка: '{s}' не число")
    else:
        print(f"Успех: x = {x}")
    finally:
        print("--- конец попытки ---")

full_demo("42")
full_demo("abc")

Успех: x = 42
--- конец попытки ---
Ошибка: 'abc' не число
--- конец попытки ---


## 2.4. `raise` — создание исключения

Иногда нужно **самому** прервать выполнение с ошибкой, если данные некорректны:

    def set_age(age):
        if age < 0 or age > 150:
            raise ValueError(f"Некорректный возраст: {age}")
        return age

    set_age(25)     # ok
    set_age(-5)     # ValueError: Некорректный возраст: -5

**Зачем:** чтобы вызывающий код **знал**, что что-то не так, и мог это обработать. Это лучше, чем молча вернуть `None` или напечатать сообщение.

### `raise` можно перехватить

    try:
        set_age(-5)
    except ValueError as e:
        print("Поймали:", e)

In [ ]:
# Пример 7. raise
def set_age(age):
    if age < 0 or age > 150:
        raise ValueError(f"Некорректный возраст: {age}")
    return age

print("set_age(25) =", set_age(25))

try:
    set_age(-5)
except ValueError as e:
    print("Поймали:", e)

set_age(25) = 25
Поймали: Некорректный возраст: -5


## Практика 5. Безопасное приведение к числу

**Описание:** напишите функцию `safe_int(s)`, которая пытается привести строку `s` к целому числу. Если не получается — возвращает `None`.

**Ожидаемый результат:**
```
✅ Задание пройдено: safe_int('42') = 42
✅ Задание пройдено: safe_int('abc') = None
```

In [ ]:
# Практика 5. safe_int(s)
def safe_int(s):
    # YOUR CODE HERE
    pass


# --- Автопроверка ---
check_value("safe_int('42')", safe_int("42"), 42)
check_value("safe_int('abc')", safe_int("abc"), None)
check_value("safe_int('3.14')", safe_int("3.14"), None)

## Практика 6. Безопасное чтение файла

**Описание:** напишите функцию `read_file_safe(path)`, которая:
- если файл существует — возвращает его содержимое (строку);
- если файла нет — возвращает `None`.

Используйте `try / except FileNotFoundError`.

**Ожидаемый результат:**
```
✅ Задание пройдено: read_file_safe('sample_text.txt') не None
✅ Задание пройдено: read_file_safe('no_such.txt') = None
```

In [ ]:
# Практика 6. read_file_safe(path)
def read_file_safe(path):
    # YOUR CODE HERE
    pass


# --- Автопроверка ---
result = read_file_safe("sample_text.txt")
print("✅ Задание пройдено: read_file_safe('sample_text.txt') не None" if result is not None else "❌ Ожидалось содержимое")
check_value("read_file_safe('no_such.txt')", read_file_safe("no_such.txt"), None)

## Практика 7. Проверка с `raise`

**Описание:** напишите функцию `check_score(score)`, которая:
- если `0 <= score <= 100` — возвращает `score`;
- иначе — **вызывает исключение** `ValueError` с сообщением `"Некорректный балл"`.

**Ожидаемый результат:**
```
✅ check_score(85) = 85
✅ check_score(150) вызвал ValueError
✅ check_score(-3) вызвал ValueError
```

In [ ]:
# Практика 7. check_score(score)
def check_score(score):
    # YOUR CODE HERE
    pass


# --- Автопроверка ---
check_value("check_score(85)", check_score(85), 85)

for bad in [150, -3]:
    try:
        check_score(bad)
        print(f"❌ check_score({bad}) не вызвал ValueError")
    except ValueError as e:
        print(f"✅ check_score({bad}) вызвал ValueError: {e}")

---

# Часть 3. JSON

## 3.1. Что такое JSON

**JSON** (JavaScript Object Notation) — текстовый формат для хранения структурированных данных. Появился в JavaScript, но стал **стандартом для всех языков**.

### 3.1.1. Пример JSON-документа

    {
      "name": "Anna",
      "age": 25,
      "level": "B2",
      "scores": [85, 90, 80],
      "active": true,
      "notes": null
    }

### 3.1.2. Соответствие JSON ↔ Python

| JSON | Python |
|------|--------|
| object `{...}` | `dict` |
| array `[...]` | `list` |
| string `"..."` | `str` |
| number `42`, `3.14` | `int`, `float` |
| `true` / `false` | `True` / `False` |
| `null` | `None` |

### 3.1.3. Зачем JSON в нашем курсе

- сохранять словари слов, карточки, настройки;
- обмениваться данными с другими программами (в т. ч. с будущими ИИ-сервисами);
- читать готовые датасеты.

**JSON читается человеком** — в отличие от бинарных форматов, его можно открыть в блокноте и увидеть структуру.

## 3.2. `json.dumps` и `json.loads` — работа со строками

Модуль `json` — часть стандартной библиотеки, ничего ставить не нужно.

    import json

    # Python → JSON-строка
    text = json.dumps({"name": "Anna", "age": 25})
    # '{"name": "Anna", "age": 25}'

    # JSON-строка → Python
    obj = json.loads('{"name": "Anna"}')
    # {'name': 'Anna'}

### Параметры `json.dumps`

| Параметр | Что делает |
|----------|------------|
| `indent=2` | Красивый отступ (для чтения человеком) |
| `ensure_ascii=False` | Не экранировать русские буквы |
| `sort_keys=True` | Сортировать ключи по алфавиту |

**Рекомендация:** используйте `indent=2, ensure_ascii=False` при выводе для человека.

In [ ]:
# Пример 8. dumps и loads
import json

data = {"name": "Anna", "age": 25, "scores": [85, 90, 80]}

# Python → строка
text = json.dumps(data, indent=2, ensure_ascii=False)
print(text)
print("Тип:", type(text).__name__)

# Строка → Python
restored = json.loads(text)
print("Восстановлено:", restored)
print("Тип:", type(restored).__name__)

{
  "name": "Anna",
  "age": 25,
  "scores": [
    85,
    90,
    80
  ]
}
Тип: str
Восстановлено: {'name': 'Anna', 'age': 25, 'scores': [85, 90, 80]}
Тип: dict


## 3.3. `json.dump` и `json.load` — работа с файлами

### 3.3.1. Запись в файл

    with open("data.json", "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

### 3.3.2. Чтение из файла

    with open("data.json", "r", encoding="utf-8") as f:
        data = json.load(f)

**Запомните разницу:**

| Работа со строками | Работа с файлами |
|---------------------|-------------------|
| `json.dumps` / `json.loads` (с буквой **s** — *string*) | `json.dump` / `json.load` (без **s**) |

In [ ]:
# Пример 9. dump и load
import json

data = {
    "student": "Anna",
    "level": "B2",
    "scores": [85, 90, 80],
    "active": True,
}

# Запись
with open("student.json", "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

# Чтение
with open("student.json", "r", encoding="utf-8") as f:
    loaded = json.load(f)

print("Прочитано из файла:")
print(loaded)
print("Средний балл:", sum(loaded["scores"]) / len(loaded["scores"]))

Прочитано из файла:
{'student': 'Anna', 'level': 'B2', 'scores': [85, 90, 80], 'active': True}
Средний балл: 85.0


## 3.4. Типичные ошибки при работе с JSON

| Ошибка | Что происходит |
|--------|----------------|
| `json.load` без `open` | `AttributeError` |
| Запись через `json.dumps` в файл | В файле окажется строка, а не JSON-объект (иногда работает, но семантически неверно) |
| Ключи с одинарными кавычками | Невалидный JSON: только `"` |
| `True` / `False` / `None` в JSON-строке | Нужны `true` / `false` / `null` (автоматически при `dumps`) |
| Запятая после последнего элемента | `JSONDecodeError` |
| Комментарии `//` в JSON | Недопустимы — в JSON нет комментариев |
| Не указан `ensure_ascii=False` | Русский текст выглядит как `\u0410\u043d...` |


## Практика 8. Сохранить словарь в JSON-файл

**Описание:** дан словарь `en_ru`. Сохраните его в файл `dict.json`.

**Ожидаемый результат:**
```
✅ JSON-файл dict.json корректен
```

In [ ]:
# Практика 8. Сохранить словарь в JSON
import json

en_ru = {"hello": "привет", "cat": "кот", "book": "книга"}

# Введите ваш код ниже:
# YOUR CODE HERE


# --- Автопроверка ---
check_json_file("dict.json", en_ru)

## Практика 9. Прочитать JSON и извлечь данные

**Описание:** прочитайте файл `student.json` (создан в примере 9) и извлеките средний балл из поля `scores`. Сохраните результат в переменную `avg`.

**Ожидаемый результат:**
```
✅ Задание пройдено: avg = 85.0
```

In [ ]:
# Практика 9. Чтение JSON
import json

avg = 0

# Введите ваш код ниже:
# YOUR CODE HERE


# --- Автопроверка ---
check_value("avg", avg, 85.0)

## Практика 10. Безопасное чтение JSON

**Описание:** напишите функцию `load_json_safe(path)`, которая:
- если файл существует и JSON корректен — возвращает Python-объект;
- если файла нет или JSON некорректен — возвращает `None`.

Используйте `try / except (FileNotFoundError, json.JSONDecodeError)`.

**Ожидаемый результат:**
```
✅ load_json_safe('student.json') не None
✅ load_json_safe('no_such.json') = None
```

In [ ]:
# Практика 10. load_json_safe(path)
import json

def load_json_safe(path):
    # YOUR CODE HERE
    pass


# --- Автопроверка ---
result = load_json_safe("student.json")
print("✅ load_json_safe('student.json') не None" if result is not None else "❌ Ожидался объект")
check_value("load_json_safe('no_such.json')", load_json_safe("no_such.json"), None)

---

# Часть 4. CSV

## 4.1. Что такое CSV

**CSV** (Comma-Separated Values) — текстовый формат таблицы: каждая строка — ряд, значения разделены запятой.

    name,score
    Anna,85
    Ivan,72
    Olga,91

**Где пригодится:**

- оценки студентов;
- результаты тестов;
- словари в табличном виде;
- экспорт из Excel.

**Преимущества:** простой, читается человеком, открывается в Excel / Google Sheets.

**Недостатки:** нет типов (всё строки), сложно хранить вложенные структуры.

## 4.2. Модуль `csv` — базовое чтение и запись

### 4.2.1. Чтение через `csv.reader`

    import csv

    with open("scores.csv", "r", encoding="utf-8", newline="") as f:
        reader = csv.reader(f)
        for row in reader:
            print(row)   # каждая строка — список строк

**Важно:** используйте `newline=""` при открытии — иначе на Windows появятся пустые строки.

### 4.2.2. Запись через `csv.writer`

    with open("out.csv", "w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["name", "score"])
        writer.writerow(["Anna", 85])

`writerow` записывает одну строку, `writerows` — список строк.

In [ ]:
# Пример 10. Чтение CSV
import csv

with open("scores.csv", "r", encoding="utf-8", newline="") as f:
    reader = csv.reader(f)
    for row in reader:
        print(row)

In [ ]:
# Пример 11. Запись CSV
import csv

with open("out.csv", "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["word", "length"])
    writer.writerow(["apple", 5])
    writer.writerow(["cat", 3])

print("Файл out.csv создан.")
with open("out.csv", encoding="utf-8") as f:
    print(f.read())

## 4.3. `DictReader` и `DictWriter`

### 4.3.1. `DictReader` — строки как словари

    with open("scores.csv", "r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            print(row)   # {'name': 'Anna', 'score': '85'}

Первая строка автоматически становится **ключами**.

### 4.3.2. `DictWriter` — запись из словарей

    fields = ["name", "score"]
    with open("out.csv", "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        writer.writerow({"name": "Anna", "score": 85})

**Внимание:** значения в CSV всегда строки. Если нужны числа — приводите через `int()` / `float()`.


In [ ]:
# Пример 12. DictReader
import csv

with open("scores.csv", "r", encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f)
    for row in reader:
        score = int(row["score"])
        print(f"{row['name']}: {score} (тип: {type(score).__name__})")

In [ ]:
# Пример 13. DictWriter
import csv

students = [
    {"name": "Anna", "score": 85},
    {"name": "Ivan", "score": 72},
    {"name": "Olga", "score": 91},
]

with open("students_out.csv", "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["name", "score"])
    writer.writeheader()
    writer.writerows(students)

with open("students_out.csv", encoding="utf-8") as f:
    print(f.read())

## Практика 11. Средний балл из CSV

**Описание:** прочитайте файл `scores.csv` через `csv.DictReader` и вычислите средний балл всех студентов. Сохраните результат в переменную `avg`.

**Ожидаемый результат:**
```
✅ Задание пройдено: avg = 82.66666666666667
```

In [ ]:
# Практика 11. Средний балл
import csv

avg = 0

# Введите ваш код ниже:
# YOUR CODE HERE


# --- Автопроверка ---
check_value("avg", avg, 248 / 3)

## Практика 12. Отчёт в CSV

**Описание:** дан список студентов. Запишите его в файл `report.csv` с заголовком `name,score`. Используйте `csv.DictWriter`.

**Ожидаемый результат:**
```
✅ Отчёт report.csv создан корректно
```

In [ ]:
# Практика 12. Отчёт в CSV
import csv

students = [
    {"name": "Anna", "score": 85},
    {"name": "Ivan", "score": 72},
    {"name": "Olga", "score": 91},
]

# Введите ваш код ниже:
# YOUR CODE HERE


# --- Автопроверка ---
import csv as _csv
if not os.path.exists("report.csv"):
    print("❌ Файл report.csv не создан")
else:
    with open("report.csv", "r", encoding="utf-8", newline="") as f:
        rows = list(_csv.DictReader(f))
    expected = [
        {"name": "Anna", "score": "85"},
        {"name": "Ivan", "score": "72"},
        {"name": "Olga", "score": "91"},
    ]
    if rows == expected:
        print("✅ Отчёт report.csv создан корректно")
    else:
        print("❌ Содержимое report.csv не совпадает")
        print("   Получено:", rows)

---

# Часть 5. Типичные ошибки

| Ошибка | Что происходит | Как исправить |
|--------|----------------|----------------|
| Забыли `encoding="utf-8"` | На Windows русский текст читается как `\xef\xbb\xbf` | Всегда указывать `encoding="utf-8"` |
| Режим `w` вместо `a` | Файл перезаписывается, старые данные теряются | `a` для дописывания |
| Забыли `newline=""` при CSV | Пустые строки в файле | `open(..., newline="")` |
| `json.load` вместо `json.loads` | `AttributeError` при работе со строкой | Различайте функции с `s` и без |
| `except Exception: pass` | Скрывает все ошибки | Обрабатывайте конкретные типы |
| Чтение `int(row["score"])` без try | `ValueError` на пустой строке | Проверяйте или оборачивайте в try |
| `f.write(42)` | `TypeError`: только строки | Приводите: `f.write(str(42))` |
| Полагаться на порядок строк в CSV | Порядок зависит от записи | Не полагайтесь без явной сортировки |
| Запись `json.dumps` в файл через `write` | Работает, но два шага вместо одного | Используйте `json.dump` |
| Создание файла, а затем чтение до записи | Пустой файл или `FileNotFoundError` | Сначала `write`, потом `read` |


---

# Часть 6. Домашние задания (базовые)

## Задание 1. Записать студентов в JSON

**Описание:** дан список словарей `students`. Сохраните его в файл `students.json`.

**Ожидаемый результат:**
```
✅ JSON-файл students.json корректен
```

In [ ]:
# Задание 1. Запись студентов в JSON
import json

students = [
    {"name": "Anna", "score": 85, "level": "B2"},
    {"name": "Ivan", "score": 72, "level": "B1"},
    {"name": "Olga", "score": 91, "level": "C1"},
]

# Введите ваш код ниже:
# YOUR CODE HERE


# --- Автопроверка ---
check_json_file("students.json", students)

## Задание 2. Прочитать студентов и вывести

**Описание:** прочитайте файл `students.json` (создан в задании 1) и сохраните имя первого студента в переменную `first_name`.

**Ожидаемый результат:**
```
✅ Задание пройдено: first_name = 'Anna'
```

In [ ]:
# Задание 2. Чтение студентов
import json

first_name = ""

# Введите ваш код ниже:
# YOUR CODE HERE


# --- Автопроверка ---
check_value("first_name", first_name, "Anna")

## Задание 3. Дневник (дописывание)

**Описание:** создайте файл `diary.txt` и запишите в него две строки: `"Day 1: started Python"` и `"Day 2: learned files"`. Затем **допишите** третью строку `"Day 3: finished homework"`.

**Ожидаемый результат:**
```
✅ Содержимое файла diary.txt совпадает с ожидаемым
```

In [ ]:
# Задание 3. Дневник
# Введите ваш код ниже:
# YOUR CODE HERE


# --- Автопроверка ---
check_file_content(
    "diary.txt",
    [
        "Day 1: started Python",
        "Day 2: learned files",
        "Day 3: finished homework",
    ],
)

## Задание 4. Подсчёт строк в CSV

**Описание:** прочитайте файл `scores.csv` и посчитайте количество студентов (без строки заголовка). Сохраните в переменную `n_students`.

**Ожидаемый результат:**
```
✅ Задание пройдено: n_students = 3
```

In [ ]:
# Задание 4. Количество студентов
import csv

n_students = 0

# Введите ваш код ниже:
# YOUR CODE HERE


# --- Автопроверка ---
check_value("n_students", n_students, 3)

## Задание 5. Фильтрация CSV по баллу

**Описание:** прочитайте `scores.csv` и запишите в новый файл `top_students.csv` только студентов с баллом **больше 80**. Формат — тот же, что в исходном файле (заголовок + строки).

**Ожидаемый результат:**
```
✅ top_students.csv содержит только Anna (85) и Olga (91)
```

In [ ]:
# Задание 5. Фильтрация
import csv

# Введите ваш код ниже:
# YOUR CODE HERE


# --- Автопроверка ---
if not os.path.exists("top_students.csv"):
    print("❌ Файл top_students.csv не создан")
else:
    with open("top_students.csv", "r", encoding="utf-8", newline="") as f:
        rows = list(csv.DictReader(f))
    names = sorted(r["name"] for r in rows)
    if names == ["Anna", "Olga"]:
        print("✅ top_students.csv содержит только Anna (85) и Olga (91)")
    else:
        print("❌ Содержимое не совпадает. Получено:", rows)

---

# Часть 7. Задания «со звёздочкой»

## Задание *1. Безопасное копирование файла

**Описание:** напишите функцию `safe_copy(src, dst)`, которая:
- если исходный файл существует — копирует его содержимое в `dst` и возвращает `True`;
- если исходного файла нет — возвращает `False`.

**Ожидаемый результат:**
```
✅ safe_copy работает корректно
```

In [ ]:
# Задание *1. safe_copy(src, dst)
def safe_copy(src, dst):
    # YOUR CODE HERE
    pass


# --- Автопроверка ---
result = safe_copy("sample_text.txt", "copy.txt")
assert result is True, "safe_copy должен вернуть True"
with open("sample_text.txt", encoding="utf-8") as f1, open("copy.txt", encoding="utf-8") as f2:
    assert f1.read() == f2.read(), "Содержимое должно совпадать"

assert safe_copy("no_such.txt", "x.txt") is False, "safe_copy должен вернуть False"
print("✅ safe_copy работает корректно")

## Задание *2. Объединение двух JSON-файлов

**Описание:** даны два JSON-файла `part1.json` и `part2.json`, каждый содержит список строк. Напишите функцию `merge_json(path1, path2, out)`, которая читает оба файла, объединяет списки и сохраняет результат в `out`.

**Ожидаемый результат:**
```
✅ merge_json работает корректно
```

In [ ]:
# Задание *2. merge_json
import json

# Подготовим данные
with open("part1.json", "w", encoding="utf-8") as f:
    json.dump(["apple", "banana"], f)
with open("part2.json", "w", encoding="utf-8") as f:
    json.dump(["cherry", "date"], f)

def merge_json(path1, path2, out):
    # YOUR CODE HERE
    pass


# --- Автопроверка ---
merge_json("part1.json", "part2.json", "merged.json")
with open("merged.json", encoding="utf-8") as f:
    data = json.load(f)
assert data == ["apple", "banana", "cherry", "date"], f"Получено: {data}"
print("✅ merge_json работает корректно")

## Задание *3. CSV → JSON

**Описание:** прочитайте `scores.csv` и сохраните все строки в JSON-файл `scores.json`. Значения `score` должны быть **числами** (`int`), не строками.

**Ожидаемый результат:**
```
✅ scores.json содержит корректные данные
```

In [ ]:
# Задание *3. CSV → JSON
import csv, json

# Введите ваш код ниже:
# YOUR CODE HERE


# --- Автопроверка ---
with open("scores.json", encoding="utf-8") as f:
    data = json.load(f)
expected = [
    {"name": "Anna", "score": 85},
    {"name": "Ivan", "score": 72},
    {"name": "Olga", "score": 91},
]
assert data == expected, f"Получено: {data}"
print("✅ scores.json содержит корректные данные")

## Задание *4. Своё исключение

**Описание:** создайте собственное исключение `EmptyTextError` (наследник `Exception`) и напишите функцию `analyze_text(text)`, которая:
- если `text` пустая или состоит из пробелов — **вызывает** `EmptyTextError`;
- иначе возвращает количество слов в тексте.

**Ожидаемый результат:**
```
✅ analyze_text корректно обрабатывает оба случая
```

In [ ]:
# Задание *4. Своё исключение
class EmptyTextError(Exception):
    pass

def analyze_text(text):
    # YOUR CODE HERE
    pass


# --- Автопроверка ---
assert analyze_text("AI helps learn English") == 4
for bad in ["", "   ", "\n\t"]:
    try:
        analyze_text(bad)
        print(f"❌ analyze_text({bad!r}) не вызвал EmptyTextError")
    except EmptyTextError:
        pass
print("✅ analyze_text корректно обрабатывает оба случая")

---

# Итоги модуля

Мы разобрали:

- **файлы**: `open()`, режимы `r`/`w`/`a`, `with`, кодировки, четыре способа чтения, запись и дописывание;
- **исключения**: `try / except / else / finally`, `raise`, типы ошибок;
- **JSON**: `dumps`/`loads`, `dump`/`load`, `indent`, `ensure_ascii`;
- **CSV**: `csv.reader`, `csv.writer`, `DictReader`, `DictWriter`, `newline=""`;
- **типичные ошибки** и способы их избежать.

**Что дальше:** в модуле 8 перейдём к регулярным выражениям и расширенным возможностям `collections`.

---

# Литература

### Основная

1. **Лутц М.** Изучаем Python. 5-е изд. — Гл. 9 «Кортежи, файлы и прочее».
2. **Свейгарт А.** Автоматизация рутинных задач с помощью Python. 2-е изд. — Гл. 8 «Чтение и запись файлов», гл. 9 «Организация файлов».
3. **Документация Python (рус.).** Файлы: https://docs.python.org/ru/3/tutorial/inputoutput.html#reading-and-writing-files
4. **Документация Python (рус.).** Исключения: https://docs.python.org/ru/3/tutorial/errors.html
5. **Документация Python (рус.).** `json`: https://docs.python.org/ru/3/library/json.html
6. **Документация Python (рус.).** `csv`: https://docs.python.org/ru/3/library/csv.html

### Дополнительная

1. **Real Python.** Reading and Writing Files in Python: https://realpython.com/read-write-files-python/
2. **Real Python.** Python Exceptions: An Introduction: https://realpython.com/python-exceptions/
3. **Real Python.** Working With JSON Data in Python: https://realpython.com/python-json/
4. **Real Python.** Reading and Writing CSV Files: https://realpython.com/python-csv/
5. **Stepik.** Курс «Поколение Python: для начинающих» — модули «Файлы», «Исключения».
6. **W3Schools.** Python File Handling, Python JSON.

### Для практики

- **Codewars** (уровень 7 kyu) — задачи на файлы и JSON.
- **HackerRank** — раздел Python → Errors and Exceptions.